In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 
from PIL import Image
import os .path
import os
import matplotlib.image as mpimg
import numpy as np

In [2]:
x=[]
def load_images(path, classes):
    for fol in classes:
        seq=os.listdir(path+u'/'+fol)
        for img in seq:
            if(img=='T1w'):
                imgg=img
                im=os.listdir(path+u'/'+fol+u'/'+imgg)
                for mri in im:
                    pat=path+u'/'+fol+u'/'+imgg+u'/'+mri
                    fimg=mpimg.imread(pat);
                    x.append(fimg)
    print(len(x))
    return x

In [3]:
path1 = "/kaggle/input/rsna-final-36-v14/Patients_New36/Patients_New36"
path2 = "/kaggle/input/rsna-final-36-v14/train4/train4"
path3 = "/kaggle/input/rsna-final-36-v14/train4_version3/train4_version3"
path4 = "/kaggle/input/rsna-final-36-v14/Dataset_Four(36_Patients)/Dataset_Four(36_Patients)"

classes = os.listdir(path1) 
classes2= os.listdir(path2)  
classes3= os.listdir(path3)
classes4= os.listdir(path4)

images = load_images(path1, classes) 
images = load_images(path2, classes2)  
images=  load_images(path3, classes3)
# images=  load_images(path4, classes4)
images = np.array(images)  # Convert the list of images to a NumPy array


# Now you can access the shape of the images array
print(images.shape)
x=images
# 116 Patients

795
1444
4950
(4950, 256, 256)


In [4]:
def split_and_normalize_data(x):
    """
    Shuffles the input data array and splits it into training and testing sets.
    
    Parameters:
        x (numpy.ndarray): The input data array.

    Returns:
        tuple: A tuple containing the shuffled training and testing data arrays.
    """
    n = x.shape[0]
    randomize = np.arange(n)
    np.random.shuffle(randomize)
    x = x[randomize]
    test_split = round(n * 8 / 10)
    x_train = x[:test_split]
    x_test = x[test_split:]
    
    x_train = x[:test_split].astype('float32') / 255
    x_test = x[test_split:].astype('float32') / 255
    
    x_train = x_train.reshape((len(x_train), np.prod(x_train.shape[1:])))
    x_test = x_test.reshape((len(x_test), np.prod(x_test.shape[1:])))
    
    return x_train, x_test

In [5]:
x_train, x_test =split_and_normalize_data(x)

In [6]:
print(x_train.shape)
print(x_test.shape)

(3960, 65536)
(990, 65536)


In [7]:
import numpy as np

def add_noise_and_clip(data_array, max_noise=0.9):
    """
    Adds random noise to the input data array and clips the values to be within the range [0, 1].

    Parameters:
        data_array (numpy.ndarray): The input data array to which noise will be added.
        max_noise (float): The maximum magnitude of the random noise. Default is 0.9.

    Returns:
        numpy.ndarray: The data array with added noise and clipped values.
    """
    noise = np.random.rand(*data_array.shape) * max_noise
    noisy_data = data_array + noise
    noisy_data = np.clip(noisy_data, 0., 1.)
    return noisy_data


# Step 3: Add noise and clip the training and validation data
x_train_noisy = add_noise_and_clip(x_train)
x_val_noisy = add_noise_and_clip(x_test)


In [8]:
def plot(x_train, p, labels=False):
    plt.figure(figsize=(17, 14))
    for i in range(150):
        ax = plt.subplot(15,10, i + 1)
        plt.imshow(x_train[i].reshape(256, 256), cmap='gray') 
        plt.xticks([])
        plt.yticks([])
        plt.gray()
        if labels:
            plt.xlabel(np.argmax(p[i]))
    plt.tight_layout()
    plt.show() 
    return 
# plot(x_train, None) 

In [9]:
from tensorflow.keras.layers import Dense, Input 
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model 
input_image = Input(shape = (x_train.shape[1], ) ) 
print(input_image.shape)

encoded = Dense(512, activation = 'LeakyReLU')(input_image)  
encoded = Dense(256, activation = 'LeakyReLU')(encoded) 
encoded = Dense(128, activation = 'LeakyReLU')(encoded) 
encoded = Dense(64, activation = 'LeakyReLU')(encoded) 
encoded = Dense(32, activation = 'LeakyReLU')(encoded) 

decoded = Dense(32, activation = 'LeakyReLU')(encoded) 
decoded = Dense(64, activation = 'LeakyReLU')(decoded)
decoded = Dense(128, activation = 'LeakyReLU')(decoded)
decoded = Dense(256, activation = 'LeakyReLU')(decoded)
decoded = Dense(512, activation = 'LeakyReLU')(decoded)
decoded = Dense(x_train.shape[1], activation = 'sigmoid')(decoded) 
autoencoder = Model(input_image, decoded) 
autoencoder.summary()

(None, 65536)
Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         [(None, 65536)]           0         
_________________________________________________________________
dense (Dense)                (None, 512)               33554944  
_________________________________________________________________
dense_1 (Dense)              (None, 256)               131328    
_________________________________________________________________
dense_2 (Dense)              (None, 128)               32896     
_________________________________________________________________
dense_3 (Dense)              (None, 64)                8256      
_________________________________________________________________
dense_4 (Dense)              (None, 32)                2080      
_________________________________________________________________
dense_5 (Dense)              (None, 32)        

In [10]:
def dice_loss(y_true, y_pred):
    numerator = tf.reduce_sum(tf.square(y_true - y_pred))
    denominator = tf.reduce_sum(tf.square(y_true)) + tf.reduce_sum(tf.square(y_pred))
    dice_loss = numerator / denominator
    return dice_loss

In [11]:
opt = keras.optimizers.Adam(learning_rate=0.001)
autoencoder.compile(loss= 'MSE' , optimizer ='adam',metrics=['accuracy']) 

In [12]:
import tensorflow as tf 
checkpointer = tf.keras.callbacks.ModelCheckpoint('/kaggle/working/model_for_PhaseOne.h5', verbose=1, save_best_only=True)
csv_logger = tf.keras.callbacks.CSVLogger('/kaggle/working/history().csv', separator=",", append=False)
callbacks = [
        tf.keras.callbacks.EarlyStopping(patience=14, monitor='val_loss'),
        tf.keras.callbacks.TensorBoard(log_dir='logs')]
history = autoencoder.fit(x_train_noisy, x_train, batch_size=14, epochs=100,callbacks=[checkpointer,csv_logger],
                          shuffle = True, validation_data=(x_val_noisy,x_test))
# history = autoencoder.fit(x_train_noisy, x_train, batch_size=14, epochs=100,shuffle = True, validation_split=0.2)

Epoch 1/100
283/283 [==============================] - 11s 31ms/step - loss: 0.0384 - accuracy: 0.0000e+00 - val_loss: 0.0269 - val_accuracy: 0.0000e+00

Epoch 00001: val_loss improved from inf to 0.02686, saving model to /kaggle/working/model_for_PhaseOne.h5
Epoch 2/100
283/283 [==============================] - 7s 26ms/step - loss: 0.0270 - accuracy: 2.5253e-04 - val_loss: 0.0268 - val_accuracy: 0.0010

Epoch 00002: val_loss improved from 0.02686 to 0.02678, saving model to /kaggle/working/model_for_PhaseOne.h5
Epoch 3/100
283/283 [==============================] - 7s 26ms/step - loss: 0.0253 - accuracy: 0.0000e+00 - val_loss: 0.0247 - val_accuracy: 0.0010

Epoch 00003: val_loss improved from 0.02678 to 0.02467, saving model to /kaggle/working/model_for_PhaseOne.h5
Epoch 4/100
283/283 [==============================] - 8s 27ms/step - loss: 0.0238 - accuracy: 2.5253e-04 - val_loss: 0.0232 - val_accuracy: 0.0000e+00

Epoch 00004: val_loss improved from 0.02467 to 0.02318, saving model 

In [25]:
preds = autoencoder.predict(x_val_noisy)
print(preds.shape)

(1222, 65536)


In [ ]:
import numpy as np

def gaussian_kernel(X, Y, sigma):
    X_sqnorm = np.sum(X ** 2, axis=1)
    Y_sqnorm = np.sum(Y ** 2, axis=1)
    dist = X_sqnorm[:, None] - 2.0 * np.dot(X, Y.T) + Y_sqnorm
    kernel = np.exp(-dist / (2 * sigma ** 2))
    return kernel

def MMD(x, y, sigma, kernel="multiscale"):
    """Emprical maximum mean discrepancy. The lower the result
       the more evidence that distributions are the same.

    Args:
        x: first sample, distribution P
        y: second sample, distribution Q
        sigma: bandwidth for Gaussian kernel
        kernel: kernel type such as "multiscale" or "rbf"
    """
    XX, YY, XY = (np.zeros((x.shape[0], x.shape[0])), np.zeros((y.shape[0], y.shape[0])), np.zeros((x.shape[0], y.shape[0])))

    if kernel == "multiscale":
        bandwidth_range = [0.2, 0.5, 0.9, 1.3]
        for a in bandwidth_range:
            K_XX = gaussian_kernel(x, x, a)
            K_YY = gaussian_kernel(y, y, a)
            K_XY = gaussian_kernel(x, y, a)

            XX += a ** 2 * (a ** 2 + K_XX) ** -1
            YY += a ** 2 * (a ** 2 + K_YY) ** -1
            XY += a ** 2 * (a ** 2 + K_XY) ** -1

    elif kernel == "rbf":
        bandwidth_range = [10, 15, 20, 50]
        for a in bandwidth_range:
            K_XX = gaussian_kernel(x, x, a)
            K_YY = gaussian_kernel(y, y, a)
            K_XY = gaussian_kernel(x, y, a)

            XX += np.exp(-0.5 * K_XX / a)
            YY += np.exp(-0.5 * K_YY / a)
            XY += np.exp(-0.5 * K_XY / a)

    else:
        raise ValueError("Unsupported kernel type. Use 'multiscale' or 'rbf'.")

    n = x.shape[0]
    m = y.shape[0]

    return np.mean(XX + YY - 2. * XY)

# Example usage

sigma = 1.0
kernel_type = "multiscale"  # Replace with "rbf" for the RBF kernel

mmd_value = MMD(preds, x_test, sigma, kernel=kernel_type)
print("MMD value:", mmd_value)

In [ ]:
# !pip install tensorflow-model-remediation

In [ ]:
# # plot(x_test, None) 
# import tensorflow_model_remediation as tfmr

# mmd_loss = tfmr.min_diff.losses.MMDLoss(kernel='gaussian',predictions_transform=None,name='mmd_loss',
#         enable_summary_histogram=False)

# loss = mmd_loss(preds, x_test)
# print(loss)

In [ ]:
print("Test Image") 
plt.imshow(x_test[28].reshape(256,256))
# plt.savefig('/kaggle/working/Test'+imgg+'.png')

In [ ]:
plot(preds, None) 
print("Denoised Image") 
len(preds)
plt.imshow(preds[28].reshape(256,256))
# plt.savefig('/kaggle/working/Predicited-10_T1wCE.png')

<!-- ## def plot_loss(history, x = 'loss', y = 'val_loss'): 
    fig, ax = plt.subplots( figsize=(8,6)) 
#     plt.plot(epochs, history.history[x], 'b', label='Training loss')
#     plt.plot(epochs, history.history[y], 'g', label='Validation loss')
    ax.plot(history.history[x],'b', label='Training loss') 
    ax.plot(history.history[y], 'g', label='Validation loss') 
    plt.xlabel('Number Of Epochs',fontsize=20)
    plt.ylabel('Loss',fontsize=20)
    plt.title('Training and Validation Loss',fontsize=16)
    plt.legend(prop={'size': 14})
plt.show() 
plot_loss(history)
plt.savefig('/kaggle/working/PhaseOne-Results-Tw.png',dpi=300,bbox_inches="tight") -->

In [ ]:
import math
MSE = np.square(np.subtract(x_test,preds)).mean() 
RMSE = math.sqrt(MSE)
print("Root Mean Squared Error", RMSE)

In [ ]:
from math import log10, sqrt
import cv2
import numpy as np
  
def PSNR(x_test, preds):
    mse = np.mean((x_test - preds) ** 2)
    if(mse == 0):  # MSE is zero means no noise is present in the signal .
                  # Therefore PSNR have no importance.
        return 100
    max_pixel = 255.0
    psnr = 20 * log10(max_pixel / sqrt(mse))
    return psnr
  
def main():
#      original = cv2.imread("original_image.png")
#      compressed = cv2.imread("compressed_image.png", 1)
     value = PSNR(x_test, preds)
     print(f"PSNR value is {value} dB")
       
if __name__ == "__main__":
    main()

In [ ]:
import tensorflow as tf
# Reshape the images to have a single channel if they are grayscale
img1 = tf.expand_dims(x_test, axis=-1)
img2 = tf.expand_dims(preds, axis=-1)

# Convert the image arrays to TensorFlow tensors
img1 = tf.convert_to_tensor(img1, dtype=tf.float32)
img2 = tf.convert_to_tensor(img2, dtype=tf.float32)

# Set the maximum pixel value
max_val = 1.0  # Assuming the pixel values are normalized between 0 and 1

# Define the power factors for multi-scale SSIM
power_factors = [0.0448, 0.2856, 0.3001, 0.2363, 0.1333]

# Calculate multi-scale SSIM
mul_ssim_score = tf.image.ssim_multiscale(
    img1,
    img2,
    max_val,
    power_factors=power_factors,
    filter_size=11,
    filter_sigma=1.5,
    k1=0.01,
    k2=0.03
)
ssim_score = tf.image.ssim(
    img1,
    img2,
    max_val
)

# Print the result
print("Multi-Scale SSIM:", mul_ssim_score.numpy())
print("Single-Scale SSIM:", ssim_score.numpy())

In [ ]:
%cd /kaggle/working/
!ls

In [ ]:
from IPython.display import FileLink
FileLink(r'model_for_PhaseOne.h5')